## 模块：协程与异步编程——单线程的高效并发

**驱动问题**：我们要开发一个高性能的大模型API调用后端。它需要同时处理成百上千个用户的请求，但大部分时间其实是在等待网络响应。如果我们每次请求都创建一个新线程，内存和切换开销会非常大。协程（Coroutine）是一种更高效的方案：它能在一个线程内并发处理海量I/O任务，是现代化Web框架（如FastAPI）的核心技术。本节将掌握Python协程的核心用法。

---

### 1. 为什么需要协程？

我们来看一个常见的场景：你的程序需要同时向3个不同的URL发送请求，并汇总结果。

-   **传统方式**：逐个请求，总耗时是3次请求的时间之和。
-   **协程方式**：同时发起3个请求，在等待网络响应时（I/O阻塞），程序会去执行其他任务，总耗时接近最慢的那一次请求。

协程就是这样一个**在单线程内处理并发的利器**。它由我们程序员自己决定何时“暂停”和“切换”任务，而不是被操作系统强制打断，因此也被称为**协作式多任务**。

它最大的优势是**极轻量**，一个程序里可以同时运行成千上万个协程，而内存和CPU开销极小。非常适合**I/O密集型**任务（网络、文件读写、数据库操作）。

---

### 2. 核心语法：`async`/`await`

要使用协程，必须掌握两个新的Python关键字和事件循环。

#### 2.1 定义协程函数：`async def`

与普通函数不同，协程函数调用后**不会立即执行**，而是返回一个**协程对象**，必须交给事件循环去运行。

In [ ]:
import asyncio

# 定义一个协程函数
async def hello():
    print("Hello")
    await asyncio.sleep(1) # 模拟耗时操作，交出控制权
    print("World")

# 直接调用不会执行
h = hello()
print(type(h)) # <class 'coroutine'>

# # 必须用 asyncio.run() 来启动事件循环并执行
# asyncio.run(hello())

#### 2.2 等待协程：`await`
`await` 关键字只能在 `async def` 定义的函数内部使用。它的作用是：

1.  **暂停**当前协程。
2.  **等待**一个可等待对象（另一个协程、Task等）完成。
3.  **交出**控制权给事件循环，让它去执行其他协程。

In [ ]:
import asyncio
import time

async def say_after(delay, what):
    # asyncio.sleep 是一个协程，会在这里交出控制权
    await asyncio.sleep(delay)
    print(f"{time.strftime('%X')}: {what}")

async def main():
    print(f"开始 {time.strftime('%X')}")
    # 这里虽然写了两个await，但它们是顺序执行的，总耗时 2+1=3秒
    await say_after(2, 'world')
    await say_after(1, 'hello')
    print(f"结束 {time.strftime('%X')}")

asyncio.run(main())

**输出**：
```
开始 14:20:05
14:20:07: world
14:20:08: hello
结束 14:20:08
```
**如何让它们真正并发执行？** 这就要用到“任务”。

---

### 3. 并发执行：创建任务（Task）

要同时运行多个协程，你需要把它们包装成**任务（Task）**，然后注册到事件循环中。

#### 3.1 推荐方式：`asyncio.gather()`
这是最常用的并发方式，能同时启动多个协程，并等待它们全部完成，返回一个包含所有结果的列表。

In [ ]:
import asyncio
import time

async def worker(name, delay):
    print(f"{name} 开始")
    await asyncio.sleep(delay)
    print(f"{name} 完成")
    return f"结果_{name}"

async def main():
    # 并发执行worker("A")和worker("B")，总耗时取最长的时间（2秒）
    results = await asyncio.gather(
        worker("A", 2),
        worker("B", 1)
    )
    print(results)  # ['结果_A', '结果_B']

print(f"开始 {time.strftime('%X')}")
asyncio.run(main())
print(f"结束 {time.strftime('%X')}")

#### 3.2 更精细的控制：`asyncio.create_task()`
`create_task()` 会立即调度一个协程，并返回一个 `Task` 对象。之后你可以单独 `await` 它的结果，或者取消它。

In [ ]:
import asyncio
import time

async def worker(name, delay):
    print(f"{name} 开始")
    await asyncio.sleep(delay)
    print(f"{name} 完成")
    return f"结果_{name}"

async def main():
    # 将协程包装成任务，立即开始调度
    task_a = asyncio.create_task(worker("A", 2))
    task_b = asyncio.create_task(worker("B", 1))
    
    # 在等待A和B的同时，主协程可以继续做其他事
    print("主协程在干别的事...")
    
    # 最后再等待任务完成并获取结果
    result_a = await task_a
    result_b = await task_b
    print(result_a, result_b)

asyncio.run(main())

---

### 4. 协程同步机制：信号量（Semaphore）

虽然协程是单线程的，但在`await`处切换时，如果有**共享资源**（如全局计数器）或者需要**限制并发数量**（如爬虫频率），就需要同步工具。其中最常用的是信号量 `Semaphore`。

#### 信号量 `asyncio.Semaphore`
它像一个“令牌桶”，可以限制同时访问某个资源的协程数量。这对于控制API调用频率、限制数据库连接数等场景至关重要。

In [ ]:
import asyncio

# 创建一个信号量，最多允许2个协程同时运行
sem = asyncio.Semaphore(2)

async def limited_task(name):
    # async with sem 会在进入时获取令牌，退出时归还令牌
    async with sem:
        print(f"{name} 获得许可，开始执行")
        await asyncio.sleep(2)
        print(f"{name} 执行完毕，归还许可")

async def main():
    # 同时启动5个任务，但因为信号量限制，同时最多只有2个在执行
    await asyncio.gather(
        limited_task("任务1"),
        limited_task("任务2"),
        limited_task("任务3"),
        limited_task("任务4"),
        limited_task("任务5")
    )

asyncio.run(main())

**你会观察到**：任务1和2先获得许可，2秒后它们完成，任务3和4获得许可，最后是任务5。

---

### 5. 实战：高性能异步API客户端 (`aiohttp`)

协程最强大的应用场景就是编写高性能的网络客户端。我们以 `aiohttp` 库为例，演示如何用协程并发请求多个URL。
（`pip install aiohttp`）

In [ ]:
pip install aiohttp

In [ ]:
import asyncio
import aiohttp
import time

# 异步请求函数，接受session、url和信号量
async def fetch(session, url, sem):
    async with sem: # 使用信号量限制并发
        try:
            # 发起异步GET请求
            async with session.get(url, timeout=5) as resp:
                text = await resp.text()
                print(f"✅ {url} 状态: {resp.status}, 长度: {len(text)}")
                return len(text)
        except Exception as e:
            print(f"❌ {url} 异常: {e}")
            return 0

async def main():
    urls = [
        "https://httpbin.org/delay/1",
        "https://httpbin.org/delay/2",
        "https://httpbin.org/json",
    ]
    # 限制最多3个并发请求
    sem = asyncio.Semaphore(3)

    start = time.time()
    # 创建一个客户端会话（session），可复用底层TCP连接，性能更高
    async with aiohttp.ClientSession() as session:
        # 为每个URL创建一个fetch协程
        tasks = [fetch(session, url, sem) for url in urls]
        # 并发执行所有任务
        results = await asyncio.gather(*tasks, return_exceptions=True)
    
    print(f"总耗时: {time.time() - start:.2f} 秒")
    print("各页面大小:", results)

asyncio.run(main())

**关键点**：
-   `aiohttp.ClientSession()`：复用连接，提升性能。
-   `asyncio.gather(*tasks)`：`*` 用于将列表解包为多个参数。
-   `return_exceptions=True`：某个任务失败不影响其他任务。

---

### 6. 优质练习与答案

**练习1 – 协程超时控制**  
用 `asyncio.wait_for` 为一个慢速任务设置超时时间。

In [ ]:
import asyncio

async def slow_task():
    await asyncio.sleep(5)
    return "慢任务完成"

async def main():
    try:
        result = await asyncio.wait_for(slow_task(), timeout=2)
        print(result)
    except asyncio.TimeoutError:
        print("任务超时，已取消！")

asyncio.run(main())

**练习2 – 限制并发的批量下载器**  
编写一个函数 `batch_download(urls, max_concurrent=2)`，它使用信号量限制并发数，并发地请求所有URL并打印状态码。

In [ ]:
import asyncio
import aiohttp

async def fetch_status(session, url, sem):
    async with sem:
        async with session.get(url) as resp:
            return url, resp.status

async def batch_download(urls, max_concurrent=2):
    sem = asyncio.Semaphore(max_concurrent)
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_status(session, url, sem) for url in urls]
        results = await asyncio.gather(*tasks)
        for url, status in results:
            print(f"{url} -> 状态码: {status}")

# 测试
urls = ["https://httpbin.org/status/200"] * 4
asyncio.run(batch_download(urls, max_concurrent=2))